# ViAI-QA: how the platform works with RAG, an LLM and MCP

This notebook follows the platform end to end while it checks a Vietnamese medical RAG system:

1. **RAG**: retrieve documents for a question (character n-gram TF-IDF over 17,835 ViMedAQA passages).
2. **Tier 1 detector**: score one (document, question, answer) as `no` / `intrinsic` / `extrinsic`.
3. **One RAG turn**: split the answer into claims, check every claim against every retrieved chunk, compute faithfulness and the RAG metrics.
4. **Guard**: decide `pass` / `regenerate` / `block` before the user sees the answer.
5. **LLM in the loop**: a chatbot answers with Claude, the guard stops an unsupported sentence, and the chatbot rewrites.
6. **Tier 2 judge agent**: Claude, with tools, re-examines the claims Tier 1 is unsure about.
7. **MCP**: any agent calls ViAI-QA over the Model Context Protocol. This is a real MCP client talking to the `viai-qa-mcp` server over stdio.
8. **Feedback**: reviewer corrections go back into retraining.

<img src="../images/architecture/rag-evaluation-pipeline.png" width="85%">

| Part | In this notebook |
|---|---|
| ViAI-QA API, guard, claim splitting, RAG metrics, audits, feedback | **Real**: the API runs locally (uvicorn + SQLite) |
| TF-IDF retriever, `rag_demo` chatbot | **Real** |
| MCP client ↔ `viai-qa-mcp` server | **Real** (stdio) |
| Tier 1 detector | **Keyword mock** when run locally, because this machine has no NLI/NER/Qwen weights. Set `VIAI_QA_URL` to a deployed API to use the real detector. |
| Claude (answer generation, judge) | **Real** with `ANTHROPIC_API_KEY`. Without it, a scripted client plays Claude and the judge call is skipped. |

## 0. Start the API

In [1]:
import json, os, pathlib, socket, subprocess, sys, time

import pandas as pd
import requests

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "dvc.yaml").exists():
    ROOT = ROOT.parent
for p in ("applications/api", "applications/rag-demo", "client-sdk/python", "machine-learning"):
    sys.path.insert(0, str(ROOT / p))
pd.set_option("display.max_colwidth", 90)

API_KEY = os.getenv("VIAI_QA_API_KEY", "vihallu-dev-key-local")   # DEV_API_KEY of a non-production API
EXTERNAL_API = os.getenv("VIAI_QA_URL")                           # a deployed API with the real detector
HAS_CLAUDE = bool(os.getenv("ANTHROPIC_API_KEY"))
server = None

if EXTERNAL_API:
    BASE_URL = EXTERNAL_API.rstrip("/")
else:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0)); port = s.getsockname()[1]
    db = ROOT / "artifacts/local/notebook-demo.db"
    db.unlink(missing_ok=True)
    env = {**os.environ, "DISABLE_MODEL": "true", "DATABASE_URL": f"sqlite:///{db}", "DEV_API_KEY": API_KEY,
           "ENVIRONMENT": "development", "PYTHONPATH": os.pathsep.join([str(ROOT / "applications/api"), str(ROOT / "machine-learning")])}
    log = open(ROOT / "artifacts/local/notebook-api.log", "w")
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", str(port),
                               "--log-level", "warning"], cwd=ROOT / "applications/api", env=env, stdout=log, stderr=log)
    BASE_URL = f"http://127.0.0.1:{port}"
    for _ in range(60):
        try:
            if requests.get(f"{BASE_URL}/health", timeout=1).ok:
                break
        except requests.ConnectionError:
            time.sleep(0.5)

os.environ.update(VIAI_QA_URL=BASE_URL, VIAI_QA_API_KEY=API_KEY)
from viai_qa import ViaiQA
qa = ViaiQA(base_url=BASE_URL, api_key=API_KEY)
print("API:", BASE_URL, "| health:", requests.get(f"{BASE_URL}/health").json())
print("Tier 1:", "detector of the deployed API" if EXTERNAL_API else "mock engine (DISABLE_MODEL=true)",
      "| Claude:", "on" if HAS_CLAUDE else "off (scripted answers)")

API: http://127.0.0.1:55395 | health: {'status': 'ok', 'version': '0.1.0', 'environment': 'development', 'database': 'connected'}
Tier 1: mock engine (DISABLE_MODEL=true) | Claude: off (scripted answers)


## 1. RAG: retrieve documents

The demo chatbot's retriever (`applications/rag-demo`) is character n-gram TF-IDF over ViMedAQA passages.
On ViMedAQA test it reaches recall@3 = 0.80, against 0.31 for PhoBERT and 0.67 for LSA.

It has one known limit: very short questions share few n-grams with the right passage and can retrieve the wrong one.

In [2]:
from types import SimpleNamespace
from rag_demo.retriever import TfidfRetriever

files = [ROOT / f"data/raw/vimedaqa/vimedaqa-{s}.parquet" for s in ("train", "test")]
if all(f.exists() for f in files):
    passages = pd.concat([pd.read_parquet(f) for f in files]).drop_duplicates("context")
    docs = [SimpleNamespace(text=r.context, title=r.title, source_url=r.article_url) for r in passages.itertuples()]
else:  # no `dvc pull`: a two-passage corpus is enough to follow the notebook
    docs = [SimpleNamespace(text="Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Bài thuốc có vị đắng – tính hàn.", title="Xuyên tâm liên", source_url=""),
            SimpleNamespace(text="Cần bảo quản thuốc Comiaryl ở nơi khô thoáng, nhiệt độ dưới 30°C.", title="Comiaryl", source_url="")]

t = time.perf_counter()
retriever = TfidfRetriever(docs)
print(f"indexed {len(docs)} passages in {time.perf_counter() - t:.1f}s")

QUESTION = "Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?"
hits = retriever.search(QUESTION, k=3)
CONTEXTS = [h.text for h in hits]
pd.DataFrame([{"score": round(h.score, 3), "title": h.title, "passage": h.text[:160]} for h in hits])

indexed 17835 passages in 4.4s


,score,title,passage
0,0.496,Công dụng của từng thành phần - Xuyên tâm liên2,"Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Bài thuốc có vị đắ..."
1,0.251,Một số bài thuốc từ Mật mông hoa - Trị mắt sưng đỏ,"Mật mông hoa 9 g, Cam cúc hoa 4 g, Kinh giới 4 g, Long đởm thảo 4 g, Phòng phong 4 g, ..."
2,0.237,Những điều cần lưu ý,"Người tiêu chảy và huyết áp thấp, phụ nữ mang thai không nên dùng. Không nên dùng liên..."


## 2. Tier 1 detector: one (document, question, answer)

`POST /api/v1/audits` scores one answer against **one** document:
- `no`: the answer is grounded in the document.
- `intrinsic`: it contradicts the document.
- `extrinsic`: it adds information the document does not contain.

`breakdown` shows which signals decided: NLI, NER, the LLM probabilities, and which guardrail fired.

In [3]:
single = qa.evaluate_single(context=CONTEXTS[0], prompt=QUESTION,
                            response="Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo.")
print("label:", single["label"], "| confidence:", single["confidence"])
print("breakdown:", json.dumps({k: v for k, v in (single.get("breakdown") or {}).items() if v is not None}, ensure_ascii=False))

label: intrinsic | confidence: 0.92
breakdown: {"p_entailment": 0.05, "p_neutral": 0.05, "p_contradiction": 0.98, "new_entity_count": 0, "new_entities": [], "entity_overlap_ratio": 1.0, "vistral_extrinsic_prob": 0.05, "active_guardrail": "NLI_CONTRADICTION"}


## 3. One RAG turn: claims × chunks

`POST /api/v1/audits/rag` handles one RAG turn:
1. Split the answer into claims.
2. Score **every claim against every retrieved chunk**.
3. A claim is grounded if at least one chunk supports it.
4. Compute `faithfulness` (the share of grounded claims) and the RAG metrics: context utilization, recall when a `ground_truth` is given, and relevance when the engine has embeddings.

Every evaluation is stored as an audit.

In [4]:
ANSWER = ("Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. "
          "Vị thuốc có vị đắng, tính hàn. "
          "Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.")
audit = qa.evaluate(QUESTION, CONTEXTS, ANSWER)
print("audit:", audit["id"], "| verdict:", audit["label"], "| faithfulness:", audit.get("faithfulness"))
print("rag metrics:", json.dumps(audit.get("rag_metrics"), ensure_ascii=False))
pd.DataFrame([{"claim": c["text"], "label": c["label"], "confidence": round(c["confidence"], 2), "chunk": c["context_index"]}
              for c in audit["claims"]])

audit: 48abda52-f9c1-4535-916f-61f5ec20757a | verdict: extrinsic | faithfulness: 0.6666666666666666
rag metrics: {"answer_relevance": 0.7353160240144463, "context_relevance": [0.569939398422671, 0.3542913886332852, 0.4851722960585705], "context_utilization": 0.3333333333333333, "context_recall": null}


,claim,label,confidence,chunk
0,"Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ.",no,0.95,0
1,"Vị thuốc có vị đắng, tính hàn.",no,0.95,0
2,Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.,extrinsic,0.86,0


## 4. Guard: `pass` / `regenerate` / `block`

`POST /api/v1/audits/rag/guard` is the call a RAG application makes **before** it shows an answer:
- **`block`**: a claim contradicts the documents. The answer is wrong and possibly harmful, so it is not shown.
- **`regenerate`**: a claim is not in the documents. The application rewrites the answer without it.
- **`pass`**: every claim is grounded.

The keyword mock answers `no` for chunks that are unrelated to a claim, so it would let a contradiction through as soon as one unrelated chunk is present. The contradiction below is therefore checked against the passage it contradicts. The real detector returns neutral/extrinsic for unrelated chunks.

In [5]:
answers = {
    "grounded": "Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Vị thuốc có vị đắng, tính hàn.",
    "adds information": ANSWER,
    "contradicts": "Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ.",
}
rows = []
for kind, ans in answers.items():
    contexts = CONTEXTS[:1] if kind == "contradicts" else CONTEXTS   # see the note above
    g = qa.guard(QUESTION, contexts, ans)
    rows.append({"answer": kind, "action": g["action"], "faithfulness": g["faithfulness"],
                 "failing claims": "; ".join(c["text"] for c in g["failing_claims"]) or "-"})
pd.DataFrame(rows)

,answer,action,faithfulness,failing claims
0,grounded,pass,1.000000,-
1,adds information,regenerate,0.666667,Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.
2,contradicts,block,0.000000,"Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ."


## 5. LLM in the loop: a guarded medical RAG chatbot

`GuardedRagChat` (`applications/rag-demo`) works in three steps:
1. **Retrieve** documents for the question.
2. **Generate** with Claude Haiku 5.5 at low effort (the cheapest setting), from the documents only.
3. **Guard** the draft:
   - `regenerate`: the chatbot retries once, told which sentences had no support.
   - `block`, or Claude declines to answer: the chatbot returns a safe fallback.

Without `ANTHROPIC_API_KEY`, a scripted client plays Claude. Its first draft adds an unsupported sentence on purpose, so the retry loop shows.

In [6]:
from rag_demo.chat import GuardedRagChat
from rag_demo.generator import ClaudeGenerator

class ScriptedClaude:
    # Stands in for anthropic.Anthropic(): first draft adds an unsupported sentence, the retry drops it
    def __init__(self, *answers):
        self.answers = list(answers)
        self.messages = SimpleNamespace(create=self._create)
    def _create(self, **request):
        text = self.answers.pop(0)
        return SimpleNamespace(stop_reason="end_turn", content=[SimpleNamespace(type="text", text=text)])

claude = None if HAS_CLAUDE else ScriptedClaude(ANSWER, answers["grounded"])
chat = GuardedRagChat(retriever, ClaudeGenerator(client=claude), qa, k=3)
result = chat.ask(QUESTION)
for i, a in enumerate(result["attempts"], 1):
    print(f"attempt {i}: {a['action']:<10} faithfulness={a.get('faithfulness')}  {a['answer']}")
print("\nshown to the user:", result["answer"])

attempt 1: regenerate faithfulness=0.6666666666666666  Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Vị thuốc có vị đắng, tính hàn. Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.
attempt 2: pass       faithfulness=1.0  Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Vị thuốc có vị đắng, tính hàn.

shown to the user: Theo Đông Y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ. Vị thuốc có vị đắng, tính hàn.


## 6. Tier 2: the judge agent (Claude + tool calling)

To keep cost down, only the claims Tier 1 is unsure about go to Claude: confidence below 0.75, or a guardrail fired.

The agent has three tools:
- `detect_hallucination`: calls Tier 1 again.
- `check_numbers`: compares numbers and doses.
- `submit_verdict`: returns the verdict in a strict schema.

`POST /api/v1/audits/{id}/judge` runs the agent and stores its verdicts on the audit.

In [7]:
from backend.app.evaluation.judge_agent import is_doubtful
from backend.app.evaluation.schemas import ClaimVerdict

doubtful = [c["text"] for c in audit["claims"] if is_doubtful(ClaimVerdict(**c))]
print("claims the judge would get:", doubtful or "none (Tier 1 is confident about every claim)")
if HAS_CLAUDE:
    judged = qa.judge(audit["id"])
    print(json.dumps(judged.get("judge"), ensure_ascii=False, indent=1)[:1500])
else:
    print("(no ANTHROPIC_API_KEY: the Claude call is skipped)")

claims the judge would get: ['Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.']
(no ANTHROPIC_API_KEY: the Claude call is skipped)


## 7. MCP: any agent calls ViAI-QA over the Model Context Protocol

`viai-qa-mcp` is an MCP server that speaks stdio. It exposes ViAI-QA as tools: guard, evaluate, evaluate a dataset, deep audit, list failing claims, and submit feedback.

The cell below is a **real MCP client**:
1. Start the server.
2. List its tools.
3. Call `guard_rag_answer` and `list_failing_claims`.

Claude Desktop, Claude Code or any MCP-capable agent does exactly these steps.

In [8]:
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

params = StdioServerParameters(
    command=sys.executable, args=["-m", "viai_qa.mcp_server"],
    env={**os.environ, "PYTHONPATH": str(ROOT / "client-sdk/python")},
)

def payload(result):
    # structured output of a dict/list tool is wrapped as {"result": ...}
    data = result.structured_content or json.loads(result.content[0].text)  # mcp 2.x: snake_case fields
    return data.get("result", data) if isinstance(data, dict) else data

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as mcp:
        info = await mcp.initialize()
        tools = (await mcp.list_tools()).tools
        print("server:", info.server_info.name, "| tools:", [t.name for t in tools])
        guard = payload(await mcp.call_tool("guard_rag_answer", {"question": QUESTION, "contexts": CONTEXTS[:1], "answer": answers["contradicts"]}))
        print("\nguard_rag_answer ->", json.dumps(guard, ensure_ascii=False)[:400])
        failing = payload(await mcp.call_tool("list_failing_claims", {"limit": 5}))
        print("\nlist_failing_claims ->")
        display(pd.DataFrame(failing))

server: viai-qa | tools: ['guard_rag_answer', 'evaluate_rag_response', 'get_audit', 'deep_audit', 'evaluate_dataset', 'list_failing_claims', 'submit_feedback']

guard_rag_answer -> {"action": "block", "reason": "Câu trả lời mâu thuẫn với tài liệu nguồn", "faithfulness": 0.0, "failing_sentences": [{"sentence": "Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ.", "verdict": "intrinsic"}], "audit_id": "b01ebaa6-53ab-4f43-88d8-f8b0fd46b0b1"}

list_failing_claims ->


,audit_id,question,verdict,failing_sentences
0,b01ebaa6-53ab-4f43-88d8-f8b0fd46b0b1,"Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?",intrinsic,"[{'sentence': 'Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ.', 'v..."
1,86dd423c-2d49-4611-a8af-baa2ad7e2eac,"Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?",extrinsic,"[{'sentence': 'Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.', 'verdict': 'ex..."
2,6ae6f806-9f18-4c5f-ae29-37d5d7f12e0b,"Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?",intrinsic,"[{'sentence': 'Theo Tây y, Xuyên tâm liên được gọi là khổ đởm thảo, nhất kiến hỷ.', 'v..."
3,2f8a4b8a-97d4-4106-a2d3-967612df8fe4,"Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?",extrinsic,"[{'sentence': 'Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.', 'verdict': 'ex..."
4,48abda52-f9c1-4535-916f-61f5ec20757a,"Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất kiến hỷ, có vị gì?",extrinsic,"[{'sentence': 'Thuốc này từng được quân đội dùng rộng rãi ở châu Phi.', 'verdict': 'ex..."


## 8. Feedback and the audit trail

Every evaluation and guard call above was stored as an audit. A reviewer, or an agent through the MCP tool `submit_feedback`, confirms or corrects the label. The DVC stage `export_feedback` collects these corrections to retrain the detector, which closes the flywheel.

The Trace View in the web app (`applications/web`) shows each audit claim by claim.

In [9]:
fb = qa.feedback(audit["id"], "extrinsic", comment="the army / Africa sentence is not in the documents")
print("feedback:", fb["corrected_label"])
listing = qa.list_audits(page_size=20)
pd.DataFrame([{"id": a["id"][:8], "label": a["label"], "faithfulness": a.get("faithfulness"), "prompt": a["prompt"][:60]}
              for a in listing["items"]])

feedback: extrinsic


,id,label,faithfulness,prompt
0,b01ebaa6,intrinsic,0.000000,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
1,098ecf43,no,1.000000,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
2,86dd423c,extrinsic,0.666667,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
3,6ae6f806,intrinsic,0.000000,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
4,2f8a4b8a,extrinsic,0.666667,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
5,a5d2849e,no,1.000000,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
6,48abda52,extrinsic,0.666667,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất
7,f77c8fe2,intrinsic,NaN,Xuyên tâm liên theo Đông Y được gọi là khổ đởm thảo hay nhất


In [10]:
# stop the local API started in step 0
if server is not None:
    server.terminate(); server.wait(timeout=10)
    print("local API stopped")

local API stopped
